In [ ]:
import numpy as np
import pandas as pd

In [ ]:
credits_df = pd.read_csv("credits.csv")
movies_df = pd.read_csv("movies.csv")

In [ ]:
credits_df

In [ ]:
movies_df

In [ ]:
credits_df.head()

In [ ]:
movies_df.tail()

In [ ]:
movies_df = movies_df.merge(credits_df, on = 'title')

In [ ]:
movies_df.shape

In [ ]:
movies_df.head()

In [ ]:
movies_df.info()

In [ ]:
movies_df = movies_df[['movie_id', 'title', 'overview', 'genres','keywords', 'cast', 'crew']]

In [ ]:
movies_df.head()

In [ ]:
movies_df.info()

In [ ]:
movies_df.isnull().sum()

In [ ]:
movies_df.dropna(inplace = True)
movies = movies_df

In [ ]:
movies_df.duplicated().sum()

In [ ]:
movies_df.iloc[0].genres

In [ ]:
import ast

In [ ]:
def convert(obj):
    L=[]
    for i in ast.literal_eval(obj):
        L.append(i['name'])
    return L

In [ ]:
movies_df['genres'] = movies_df['genres'].apply(convert)
movies_df['keywords'] = movies_df['keywords'].apply(convert)
movies_df.head()

In [ ]:
def convert3(obj):
    L=[]
    counter = 0
    for i in ast.literal_eval(obj):
        if counter != 3:
            L.append(i['name'])
            counter +=1
        else:
            break
        return L

In [ ]:
movies_df['cast'] = movies_df['cast'].apply(convert3)

In [ ]:
movies_df.head()

In [ ]:
def fetch_director(obj):
    L=[]
    for i in ast.literal_eval(obj):
        if i['job']=='Director':
            L.append(i['name'])
            break
    return L

In [ ]:
movies_df['crew']=movies_df['crew'].apply(fetch_director)

In [ ]:
movies_df.head()

In [ ]:
movies=movies_df

In [ ]:
movies['overview'][0]

In [ ]:
movies['overview'] = movies['overview'].apply(lambda x:x.split())

In [ ]:
movies

In [ ]:
movies['overview'][0]

In [ ]:
movies['genres']=movies['genres'].apply(lambda x:[i.replace(" ","") for i in x])
movies['keywords']=movies['keywords'].apply(lambda x:[i.replace(" ","") for i in x])
movies['crew']=movies['crew'].apply(lambda x:[i.replace(" ","") for i in x])

In [ ]:
movies

In [ ]:
movies.dropna(axis=0, how="any", inplace=True)

In [ ]:
movies.shape

In [ ]:

movies['cast']=movies['cast'].apply(lambda x:[i.replace(" ","") for i in x])

In [ ]:
movies

In [ ]:
movies.shape

In [ ]:
movies['tags'] = movies['overview']+movies['genres']+movies['keywords']+movies['cast']+movies['crew']

In [ ]:
movies

In [ ]:
new_df = movies[['movie_id', 'title', 'tags']]

In [ ]:
new_df

In [ ]:
new_df['tags']= new_df['tags'].apply(lambda x:' '.join(x))

In [ ]:
new_df

In [ ]:
new_df['tags'][0]

In [ ]:
new_df['tags']=new_df['tags'].apply(lambda x:x.lower())

In [ ]:
new_df.head()

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=5000, stop_words='english')

In [ ]:
cv.fit_transform(new_df['tags']).toarray().shape

In [ ]:
vectors = cv.fit_transform(new_df['tags']).toarray()

In [ ]:
vectors[0]

In [ ]:
len(cv.get_feature_names_out())
#refer https://365datascience.com/question/how-to-fix-attributeerror-countvectorizer-object-has-no-attribute-get-feature-names-out/

In [ ]:
import nltk

In [ ]:
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

In [ ]:
def stem(text):
    y=[]
    for i in text.split():
        y.append(ps.stem(i))
    return " ".join(y)

In [ ]:
new_df['tags']=new_df['tags'].apply(stem)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
cosine_similarity(vectors)

In [ ]:
cosine_similarity(vectors).shape

In [ ]:
s = cosine_similarity(vectors)

In [ ]:
s[0]

In [ ]:
s[0].shape

In [ ]:
sorted(list(enumerate(s[0])), reverse=True, key=lambda x:x[1])[1:6]

In [ ]:
def recommend(movie):
    movie_index = new_df[new_df['title']==movie].index[0]
    distances = s[movie_index]
    movies_list = sorted(list(enumerate(distances)), reverse = True, key=lambda x:x[1])[1:6]
    
    for i in movies_list:
        print(new_df.iloc[i[0]].title)

In [ ]:
recommend('Avatar')

In [ ]:
recommend('Iron Man')

In [ ]:
recommend('Liar Liar')

In [ ]:
recommend('Captain America: Civil War')